# Experiment 43: 33B + Neighbourhood Window Target Encoding

Test whether leakage-safe neighbourhood target rates improve the strong Experiment 41 / 33B XGBoost pipeline.

The main new idea comes from the Kaggle solution using local target rates around continuous values. Income and commute are treated as locally smooth axes instead of only being learned through tree splits.

New feature families:
- Income neighbourhood windows: ±2, ±5, ±10, ±25, ±50, ±200
- Commute neighbourhood windows: ±1, ±3, ±10
- Group-restricted income windows: ±5, ±25 using City_Type × Current_Car_Type

Target-aware windows are generated with inner cross-fitting using only the outer-fold training data. Validation rows never contribute their own labels.

In [1]:
import gc
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

TRAIN_PATH = "../data/train.csv"
TARGET = "Will_Buy_EV"

RANDOM_SEED = 42
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3

INCOME_WINDOWS = [2, 5, 10, 25, 50, 200]
COMMUTE_WINDOWS = [1, 3, 10]
GROUP_INCOME_WINDOWS = [5, 25]

train = pd.read_csv(TRAIN_PATH)

y = train[TARGET].map({"No": 0, "Yes": 1}).astype(np.int8)
X = train.drop(columns=[TARGET, "id"]).copy()

numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Train rows:", len(X))
print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
print("Positive rate:", float(y.mean()))


Train rows: 668665
Numeric columns: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
Positive rate: 0.17464500160768098


## 1. Exact 33B identity features

These are kept because Experiment 41 showed that the exact-value target/frequency representation is one of our strongest known pipelines.

The target encoding is inner-fold cross-fitted.


In [2]:
def make_identity_key(series):
    return series.astype("string").fillna("__MISSING__")


def fit_mapping(values, target, smoothing=20):
    temp = pd.DataFrame({
        "value": values,
        "target": target.to_numpy()
    })

    global_mean = float(target.mean())

    stats = (
        temp.groupby("value", dropna=False)["target"]
        .agg(["mean", "count"])
    )

    smoothed = (
        stats["count"] * stats["mean"]
        + smoothing * global_mean
    ) / (stats["count"] + smoothing)

    return smoothed.to_dict(), global_mean


def apply_mapping(values, mapping, global_mean):
    return (
        values.map(mapping)
        .fillna(global_mean)
        .astype(float)
    )


def add_identity_features(
    X_fit,
    y_fit,
    X_apply,
    columns,
    n_splits=3,
    smoothing=20
):
    X_fit = X_fit.copy()
    X_apply = X_apply.copy()

    skf_inner = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for col in columns:
        fit_keys = make_identity_key(X_fit[col])
        apply_keys = make_identity_key(X_apply[col])

        oof_values = np.zeros(len(X_fit), dtype=float)

        for train_idx, fold_idx in skf_inner.split(X_fit, y_fit):
            mapping, global_mean = fit_mapping(
                fit_keys.iloc[train_idx],
                y_fit.iloc[train_idx],
                smoothing=smoothing
            )

            oof_values[fold_idx] = apply_mapping(
                fit_keys.iloc[fold_idx],
                mapping,
                global_mean
            ).to_numpy()

        full_mapping, full_global_mean = fit_mapping(
            fit_keys,
            y_fit,
            smoothing=smoothing
        )

        X_fit[f"{col}__identity_target"] = oof_values

        X_apply[f"{col}__identity_target"] = apply_mapping(
            apply_keys,
            full_mapping,
            full_global_mean
        ).to_numpy()

        frequencies = fit_keys.value_counts(dropna=False)

        X_fit[f"{col}__identity_frequency"] = (
            fit_keys.map(frequencies)
            .fillna(0)
            .astype(float)
            .to_numpy()
        )

        X_apply[f"{col}__identity_frequency"] = (
            apply_keys.map(frequencies)
            .fillna(0)
            .astype(float)
            .to_numpy()
        )

    return X_fit, X_apply


In [3]:
def add_digit_features(X_frame, columns):
    X_frame = X_frame.copy()

    for col in columns:
        values = pd.to_numeric(
            X_frame[col],
            errors="coerce"
        )

        integer_values = values.abs().round()

        X_frame[f"{col}__digits"] = (
            np.floor(
                np.log10(
                    integer_values.clip(lower=1)
                )
            ) + 1
        )

        divisor = 10 ** (
            X_frame[f"{col}__digits"] - 1
        )

        X_frame[f"{col}__first_digit"] = (
            integer_values / divisor
        ).fillna(0).astype(float)

        X_frame[f"{col}__first_digit"] = np.floor(
            X_frame[f"{col}__first_digit"]
        )

        X_frame[f"{col}__last_digit"] = (
            integer_values.fillna(0)
            .astype(np.int64) % 10
        )

        digit_sums = np.zeros(len(X_frame), dtype=float)
        valid = integer_values.notna()

        digit_sums[valid] = (
            integer_values.loc[valid]
            .astype(np.int64)
            .astype(str)
            .map(lambda s: sum(int(ch) for ch in s))
            .to_numpy()
        )

        digit_sums[~valid] = np.nan

        X_frame[f"{col}__digit_sum"] = digit_sums

        X_frame[f"{col}__parity"] = (
            integer_values.fillna(0)
            .astype(np.int64) % 2
        )

        X_frame[f"{col}__mod100"] = (
            integer_values.fillna(0)
            .astype(np.int64) % 100
        )

        X_frame[f"{col}__mod1000"] = (
            integer_values.fillna(0)
            .astype(np.int64) % 1000
        )

        X_frame[f"{col}__ends_zero"] = (
            integer_values.fillna(0)
            .astype(np.int64) % 10 == 0
        ).astype(np.int8)

    return X_frame


## 2. Neighbourhood target encoding

For each numeric value, calculate the target rate among training observations whose value falls inside:

`value - radius <= neighbour <= value + radius`

A prior with 20 pseudo-observations toward the training mean stabilizes sparse neighbourhoods.

The implementation uses sorting and cumulative target/count sums, making each window approximately O(n log n) for the sort and O(n) for each radius.

For the outer training rows, these features are generated with inner 3-fold cross-fitting.

For the outer validation rows, statistics come only from the outer training partition.


In [4]:
def window_rate_from_reference(
    reference_values,
    reference_target,
    query_values,
    radius,
    prior_strength=20.0,
    prior_mean=None
):
    ref = pd.to_numeric(
        pd.Series(reference_values),
        errors="coerce"
    ).to_numpy(dtype=np.float64)

    target = np.asarray(
        reference_target,
        dtype=np.float64
    )

    query = pd.to_numeric(
        pd.Series(query_values),
        errors="coerce"
    ).to_numpy(dtype=np.float64)

    if prior_mean is None:
        prior_mean = float(np.mean(target))

    valid_ref = np.isfinite(ref)

    ref_values = ref[valid_ref]
    ref_target = target[valid_ref]

    if len(ref_values) == 0:
        return np.full(
            len(query),
            prior_mean,
            dtype=np.float64
        )

    order = np.argsort(
        ref_values,
        kind="mergesort"
    )

    sorted_values = ref_values[order]
    sorted_target = ref_target[order]

    cumulative_target = np.concatenate(
        ([0.0], np.cumsum(sorted_target))
    )

    cumulative_count = np.arange(
        len(sorted_values) + 1,
        dtype=np.float64
    )

    valid_query = np.isfinite(query)

    left = np.searchsorted(
        sorted_values,
        query[valid_query] - radius,
        side="left"
    )

    right = np.searchsorted(
        sorted_values,
        query[valid_query] + radius,
        side="right"
    )

    window_target = (
        cumulative_target[right]
        - cumulative_target[left]
    )

    window_count = (
        cumulative_count[right]
        - cumulative_count[left]
    )

    result = np.full(
        len(query),
        prior_mean,
        dtype=np.float64
    )

    result[valid_query] = (
        window_target
        + prior_strength * prior_mean
    ) / (
        window_count
        + prior_strength
    )

    return result


In [5]:
def add_crossfit_window_features(
    X_fit,
    y_fit,
    X_apply,
    income_windows,
    commute_windows,
    group_income_windows,
    n_inner_folds=3
):
    X_fit = X_fit.copy()
    X_apply = X_apply.copy()

    inner_skf = StratifiedKFold(
        n_splits=n_inner_folds,
        shuffle=True,
        random_state=42
    )

    fold_indices = list(
        inner_skf.split(X_fit, y_fit)
    )

    income = pd.to_numeric(
        X_fit["Annual_Income_USD"],
        errors="coerce"
    ).to_numpy()

    commute = pd.to_numeric(
        X_fit["Daily_Commute_km"],
        errors="coerce"
    ).to_numpy()

    apply_income = pd.to_numeric(
        X_apply["Annual_Income_USD"],
        errors="coerce"
    ).to_numpy()

    apply_commute = pd.to_numeric(
        X_apply["Daily_Commute_km"],
        errors="coerce"
    ).to_numpy()

    group_fit = (
        X_fit["City_Type"].astype("string").fillna("__MISSING__")
        + "__"
        + X_fit["Current_Car_Type"].astype("string").fillna("__MISSING__")
    )

    group_apply = (
        X_apply["City_Type"].astype("string").fillna("__MISSING__")
        + "__"
        + X_apply["Current_Car_Type"].astype("string").fillna("__MISSING__")
    )

    group_fit_np = group_fit.to_numpy()
    group_apply_np = group_apply.to_numpy()

    # ---------------------------------------------------------
    # Income windows
    # ---------------------------------------------------------
    for radius in income_windows:
        name = f"win_inc_{radius}"

        values = np.zeros(
            len(X_fit),
            dtype=np.float32
        )

        for inner_train_idx, inner_valid_idx in fold_indices:
            values[inner_valid_idx] = window_rate_from_reference(
                income[inner_train_idx],
                y_fit.iloc[inner_train_idx],
                income[inner_valid_idx],
                radius=radius
            )

        X_fit[name] = values

        X_apply[name] = window_rate_from_reference(
            income,
            y_fit,
            apply_income,
            radius=radius
        ).astype(np.float32)

    # ---------------------------------------------------------
    # Commute windows
    # ---------------------------------------------------------
    for radius in commute_windows:
        name = f"win_km_{radius}"

        values = np.zeros(
            len(X_fit),
            dtype=np.float32
        )

        for inner_train_idx, inner_valid_idx in fold_indices:
            values[inner_valid_idx] = window_rate_from_reference(
                commute[inner_train_idx],
                y_fit.iloc[inner_train_idx],
                commute[inner_valid_idx],
                radius=radius
            )

        X_fit[name] = values

        X_apply[name] = window_rate_from_reference(
            commute,
            y_fit,
            apply_commute,
            radius=radius
        ).astype(np.float32)

    # ---------------------------------------------------------
    # Group-restricted income windows
    # ---------------------------------------------------------
    for radius in group_income_windows:
        name = f"win_grp_inc_{radius}"

        values = np.zeros(
            len(X_fit),
            dtype=np.float32
        )

        for inner_train_idx, inner_valid_idx in fold_indices:
            train_groups = group_fit_np[inner_train_idx]
            valid_groups = group_fit_np[inner_valid_idx]

            train_income = income[inner_train_idx]
            train_y = y_fit.iloc[inner_train_idx].to_numpy()

            valid_income = income[inner_valid_idx]

            global_mean = float(
                y_fit.iloc[inner_train_idx].mean()
            )

            result = np.full(
                len(inner_valid_idx),
                global_mean,
                dtype=np.float64
            )

            # Process each group present in the validation slice.
            for group in np.unique(valid_groups):
                ref_mask = train_groups == group
                query_mask = valid_groups == group

                if not np.any(ref_mask):
                    continue

                result[query_mask] = window_rate_from_reference(
                    train_income[ref_mask],
                    train_y[ref_mask],
                    valid_income[query_mask],
                    radius=radius,
                    prior_mean=global_mean
                )

            values[inner_valid_idx] = result

        X_fit[name] = values

        # Full outer-training statistics for apply/test/validation rows.
        global_mean = float(y_fit.mean())

        apply_result = np.full(
            len(X_apply),
            global_mean,
            dtype=np.float64
        )

        for group in np.unique(group_apply_np):
            ref_mask = group_fit_np == group
            query_mask = group_apply_np == group

            if not np.any(ref_mask):
                continue

            apply_result[query_mask] = window_rate_from_reference(
                income[ref_mask],
                y_fit.to_numpy()[ref_mask],
                apply_income[query_mask],
                radius=radius,
                prior_mean=global_mean
            )

        X_apply[name] = apply_result.astype(np.float32)

    return X_fit, X_apply


In [6]:
def build_preprocessor(X_frame):
    numeric = X_frame.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical = X_frame.select_dtypes(
        exclude=["number"]
    ).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, numeric),
        ("cat", categorical_pipeline, categorical)
    ])


## 3. Model

This starts from the known 33B XGBoost configuration.

We deliberately keep the model relatively close to the known baseline so that the experiment mainly answers one question:

**Do neighbourhood windows add useful ranking information?**



In [7]:
def train_fold(
    X_tr,
    y_tr,
    X_va,
    y_va,
    fold
):
    t0 = time.time()

    # Exact 33B identity features
    X_tr, X_va = add_identity_features(
        X_tr,
        y_tr,
        X_va,
        numeric_cols,
        n_splits=3,
        smoothing=20
    )

    # Exact 33B digit features
    X_tr = add_digit_features(
        X_tr,
        numeric_cols
    )

    X_va = add_digit_features(
        X_va,
        numeric_cols
    )

    # NEW: neighbourhood target windows
    X_tr, X_va = add_crossfit_window_features(
        X_tr,
        y_tr,
        X_va,
        income_windows=INCOME_WINDOWS,
        commute_windows=COMMUTE_WINDOWS,
        group_income_windows=GROUP_INCOME_WINDOWS,
        n_inner_folds=N_INNER_FOLDS
    )

    window_cols = [
        c for c in X_tr.columns
        if c.startswith("win_")
    ]

    print(
        f"Window features: {len(window_cols)} | "
        f"Total raw features: {X_tr.shape[1]}"
    )

    preprocessor = build_preprocessor(X_tr)

    X_tr_encoded = preprocessor.fit_transform(X_tr)
    X_va_encoded = preprocessor.transform(X_va)

    print(
        f"Encoded shape: {X_tr_encoded.shape}"
    )

    model = XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.04,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.85,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        device="cpu",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_tr_encoded,
        y_tr,
        eval_set=[(X_va_encoded, y_va)],
        verbose=False
    )

    pred = model.predict_proba(
        X_va_encoded
    )[:, 1]

    score = roc_auc_score(
        y_va,
        pred
    )

    elapsed = (time.time() - t0) / 60

    print(
        f"Fold {fold} AUC: {score:.6f} | "
        f"Time: {elapsed:.1f} min"
    )

    del (
        X_tr,
        X_va,
        X_tr_encoded,
        X_va_encoded,
        preprocessor,
        model
    )

    gc.collect()

    return pred, score


# Submission 14: Experiment 43

This submission uses the exact Experiment 43 pipeline:

- Exact 33B identity target/frequency features
- Exact 33B digit features
- 11 neighbourhood window features
- 5-fold training
- Exact Experiment 43 XGBoost parameters
- Test predictions averaged across all 5 folds


In [8]:
TEST_PATH = "../data/test.csv"
OUTPUT_PATH = "../submissions/submission_14.csv"

test = pd.read_csv(TEST_PATH)

X_test = test.drop(columns=["id"]).copy()

print("Train rows:", len(X))
print("Test rows:", len(X_test))
print("Test ID column:", "id" in test.columns)


Train rows: 668665
Test rows: 286571
Test ID column: True


In [9]:
def train_submission_fold(
    X_tr,
    y_tr,
    X_apply,
    fold
):
    t0 = time.time()

    # Exact Exp43 identity features
    X_tr, X_apply = add_identity_features(
        X_tr,
        y_tr,
        X_apply,
        numeric_cols,
        n_splits=3,
        smoothing=20
    )

    # Exact Exp43 digit features
    X_tr = add_digit_features(
        X_tr,
        numeric_cols
    )

    X_apply = add_digit_features(
        X_apply,
        numeric_cols
    )

    # Exact Exp43 neighbourhood windows
    X_tr, X_apply = add_crossfit_window_features(
        X_tr,
        y_tr,
        X_apply,
        income_windows=INCOME_WINDOWS,
        commute_windows=COMMUTE_WINDOWS,
        group_income_windows=GROUP_INCOME_WINDOWS,
        n_inner_folds=N_INNER_FOLDS
    )

    window_cols = [
        c for c in X_tr.columns
        if c.startswith("win_")
    ]

    print(
        f"Window features: {len(window_cols)} | "
        f"Total raw features: {X_tr.shape[1]}"
    )

    preprocessor = build_preprocessor(X_tr)

    X_tr_encoded = preprocessor.fit_transform(X_tr)
    X_apply_encoded = preprocessor.transform(X_apply)

    print(
        f"Encoded train shape: {X_tr_encoded.shape}"
    )

    print(
        f"Encoded apply shape: {X_apply_encoded.shape}"
    )

    # Exact Exp43 XGBoost configuration
    model = XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.04,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.85,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        device="cpu",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_tr_encoded,
        y_tr,
        verbose=False
    )

    pred = model.predict_proba(
        X_apply_encoded
    )[:, 1]

    elapsed = (time.time() - t0) / 60

    print(
        f"Fold {fold} complete | "
        f"Mean prediction: {pred.mean():.6f} | "
        f"Time: {elapsed:.1f} min"
    )

    del (
        X_tr,
        X_apply,
        X_tr_encoded,
        X_apply_encoded,
        preprocessor,
        model
    )

    gc.collect()

    return pred


In [10]:
OUTER_SKF = StratifiedKFold(
    n_splits=N_OUTER_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED
)

test_predictions = np.zeros(
    len(X_test),
    dtype=np.float64
)

submission_start = time.time()

for fold, (tr_idx, _) in enumerate(
    OUTER_SKF.split(X, y),
    start=1
):
    print("\n" + "=" * 70)
    print(f"EXP43 SUBMISSION FOLD {fold}/{N_OUTER_FOLDS}")
    print("=" * 70)

    X_tr = X.iloc[tr_idx].copy()
    y_tr = y.iloc[tr_idx]

    fold_pred = train_submission_fold(
        X_tr,
        y_tr,
        X_test.copy(),
        fold
    )

    test_predictions += (
        fold_pred / N_OUTER_FOLDS
    )

    del X_tr, y_tr, fold_pred
    gc.collect()

print("\n" + "=" * 80)
print("SUBMISSION 14")
print("=" * 80)

submission = pd.DataFrame({
    "id": test["id"],
    TARGET: test_predictions
})

submission.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())
print("Prediction min:", float(test_predictions.min()))
print("Prediction max:", float(test_predictions.max()))
print("Prediction mean:", float(test_predictions.mean()))
print(
    f"Total time: "
    f"{(time.time() - submission_start) / 60:.1f} min"
)
print("=" * 80)



EXP43 SUBMISSION FOLD 1/5
Window features: 11 | Total raw features: 94
Encoded train shape: (534932, 105)
Encoded apply shape: (286571, 105)
Fold 1 complete | Mean prediction: 0.175209 | Time: 3.6 min

EXP43 SUBMISSION FOLD 2/5
Window features: 11 | Total raw features: 94
Encoded train shape: (534932, 105)
Encoded apply shape: (286571, 105)
Fold 2 complete | Mean prediction: 0.175169 | Time: 2.4 min

EXP43 SUBMISSION FOLD 3/5
Window features: 11 | Total raw features: 94
Encoded train shape: (534932, 105)
Encoded apply shape: (286571, 105)
Fold 3 complete | Mean prediction: 0.174856 | Time: 2.4 min

EXP43 SUBMISSION FOLD 4/5
Window features: 11 | Total raw features: 94
Encoded train shape: (534932, 105)
Encoded apply shape: (286571, 105)
Fold 4 complete | Mean prediction: 0.175033 | Time: 2.3 min

EXP43 SUBMISSION FOLD 5/5
Window features: 11 | Total raw features: 94
Encoded train shape: (534932, 105)
Encoded apply shape: (286571, 105)
Fold 5 complete | Mean prediction: 0.175899 | Time